# NV center ODMR peak prediction

Predicts where the two CW-ODMR resonance peaks will appear (in GHz) for a given static field, in gauss, from the coil.

Zero-field splitting `D = 2.87 GHz` is the standard room-temperature value for the NV- ground-state triplet (matches this project's other scripts' default drive frequency, e.g. `cw_odmr.py`/`cw_odmr_lock_in.py`'s `freq_hz=2.87e9`).

**This diamond is (100)-cut, and the coil field is aligned with the cut face -- i.e. along the [100] crystal direction, which is the face's normal.** The four NV symmetry axes run along the <111> directions, NOT along [100], so the field is *not* aligned with any NV axis here. The angle between [100] and every <111> direction is the same (the "magic angle", ~54.74 degrees) by crystal symmetry, so all four NV orientations see an identical field projection -- you still only get 2 resolvable ODMR peaks (not 8), but the splitting isn't the simple `D +- gamma*B` you'd get for an on-axis field. That needs diagonalizing the spin-1 Hamiltonian instead, which is what this notebook does.

## Constants

In [1]:
import numpy as np

# Zero-field splitting, GHz.
D_ZERO_FIELD_GHZ = 2.87

# Electron gyromagnetic ratio, MHz/G. gamma_e/2pi = g_e * mu_B / h, using the
# free-electron g-factor (g_e = 2.0023193043622, CODATA) since the NV-
# center's electron spin g-factor is very close to free-electron:
#   gamma_e/2pi = 2.0023193043622 * 9.2740100783e-24 J/T / 6.62607015e-34 J*s
#               = 28.02495 GHz/T = 2.802495 MHz/G
# matches the commonly quoted NV literature value of ~2.8 MHz/G (e.g.
# Doherty et al., Phys. Rep. 528, 1 (2013)).
GAMMA_E_MHZ_PER_G = 2.802495

# Angle between the [100] cut-face normal (the coil field direction) and
# every <111> NV axis: arccos(1/sqrt(3)), the crystallographic "magic
# angle" -- same for all four NV orientations by symmetry.
MAGIC_ANGLE_DEG = np.degrees(np.arccos(1 / np.sqrt(3)))
print(f"magic angle: {MAGIC_ANGLE_DEG:.4f} deg")

magic angle: 54.7356 deg


## Spin-1 Hamiltonian

In the `|+1>, |0>, |-1>` basis (`Sz` eigenstates), with the field at angle `theta` to the NV axis (azimuthal angle doesn't matter when strain `E=0`, since the Hamiltonian is then rotationally symmetric about the NV axis):

```
H/h = [[D + gamma*B*cos(theta),  gamma*B*sin(theta)/sqrt(2),  E                      ],
       [gamma*B*sin(theta)/sqrt(2),  0,                       gamma*B*sin(theta)/sqrt(2)],
       [E,                        gamma*B*sin(theta)/sqrt(2),  D - gamma*B*cos(theta) ]]
```

`E` (strain/transverse zero-field splitting) defaults to 0 here since it's not characterized for this sample -- pass a nonzero `e_ghz` if you have a measured value. Diagonalizing this and tracking which eigenstate is the `|0>`-like one (largest overlap with the bare `|0>` state) gives the two ODMR transition frequencies as that state's energy difference to the other two eigenvalues.

In [2]:
def build_hamiltonian_ghz(field_g, angle_deg, d_ghz, e_ghz, gamma_mhz_per_g):
    theta = np.radians(angle_deg)
    gamma_b_ghz = gamma_mhz_per_g * field_g / 1000.0
    bz = gamma_b_ghz * np.cos(theta)
    bx = gamma_b_ghz * np.sin(theta)
    return np.array([
        [d_ghz + bz,     bx / np.sqrt(2), e_ghz],
        [bx / np.sqrt(2), 0.0,            bx / np.sqrt(2)],
        [e_ghz,          bx / np.sqrt(2), d_ghz - bz],
    ])


def nv_resonance_frequencies(field_g, angle_deg=MAGIC_ANGLE_DEG, d_ghz=D_ZERO_FIELD_GHZ,
                              e_ghz=0.0, gamma_mhz_per_g=GAMMA_E_MHZ_PER_G):
    """
    Predict the two ODMR resonance frequencies (GHz) for an NV center in a
    static field of field_g gauss, applied at angle_deg to the NV axis
    (default: the magic angle, i.e. field along [100] on this (100)-cut
    diamond). angle_deg=0 reduces to the simple on-axis D +- gamma*B case.
    """
    H = build_hamiltonian_ghz(field_g, angle_deg, d_ghz, e_ghz, gamma_mhz_per_g)
    energies, vectors = np.linalg.eigh(H)
    ms0_idx = np.argmax(np.abs(vectors[1, :]) ** 2)
    ref_energy = energies[ms0_idx]
    others = [energies[i] for i in range(3) if i != ms0_idx]
    f_low, f_high = sorted(abs(e - ref_energy) for e in others)
    return f_low, f_high

## Sanity checks

- At `B=0`, both predicted frequencies should just be `D` (2.87 GHz), since the field-free splitting is degenerate.
- At `angle_deg=0` (field aligned with the NV axis, e.g. if this were a (111)-cut diamond instead), the result should exactly match the simple `D -+ gamma*B` formula, since an on-axis field doesn't mix the spin states.

In [3]:
print("B=0 GHz:", nv_resonance_frequencies(0))

for g in [5, 10, 20]:
    from_hamiltonian = nv_resonance_frequencies(g, angle_deg=0)
    simple_formula = (D_ZERO_FIELD_GHZ - GAMMA_E_MHZ_PER_G * g / 1000,
                       D_ZERO_FIELD_GHZ + GAMMA_E_MHZ_PER_G * g / 1000)
    print(f"{g} G  hamiltonian(angle=0): {from_hamiltonian}  simple: {simple_formula}")

B=0 GHz: (np.float64(2.87), np.float64(2.87))
5 G  hamiltonian(angle=0): (np.float64(2.855987525), np.float64(2.884012475))  simple: (2.855987525, 2.884012475)
10 G  hamiltonian(angle=0): (np.float64(2.8419750500000003), np.float64(2.89802495))  simple: (2.8419750500000003, 2.89802495)
20 G  hamiltonian(angle=0): (np.float64(2.8139501), np.float64(2.9260499))  simple: (2.8139501, 2.9260499)


## Predicted peaks vs. field (magic angle, [100] field on this (100)-cut diamond)

Using the coil's calibrated field range from `spd1168x.py` (up to ~28 G at 2 A).

In [4]:
for g in [0, 5, 10, 15, 20, 25, 30]:
    f_low, f_high = nv_resonance_frequencies(g)
    print(f"{g:5.1f} G  ->  f_low = {f_low:.5f} GHz   f_high = {f_high:.5f} GHz   "
          f"splitting = {(f_high - f_low) * 1000:.2f} MHz")

  0.0 G  ->  f_low = 2.87000 GHz   f_high = 2.87000 GHz   splitting = 0.00 MHz
  5.0 G  ->  f_low = 2.86198 GHz   f_high = 2.87816 GHz   splitting = 16.18 MHz
 10.0 G  ->  f_low = 2.85409 GHz   f_high = 2.88645 GHz   splitting = 32.36 MHz
 15.0 G  ->  f_low = 2.84635 GHz   f_high = 2.89489 GHz   splitting = 48.54 MHz
 20.0 G  ->  f_low = 2.83874 GHz   f_high = 2.90345 GHz   splitting = 64.72 MHz
 25.0 G  ->  f_low = 2.83126 GHz   f_high = 2.91216 GHz   splitting = 80.89 MHz
 30.0 G  ->  f_low = 2.82393 GHz   f_high = 2.92100 GHz   splitting = 97.07 MHz


## Plot

In [5]:
import matplotlib.pyplot as plt

fields = np.linspace(0, 30, 200)
lows, highs = zip(*(nv_resonance_frequencies(g) for g in fields))

plt.figure(figsize=(7, 4))
plt.plot(fields, lows, label="f_low")
plt.plot(fields, highs, label="f_high")
plt.axhline(D_ZERO_FIELD_GHZ, color="gray", linestyle="--", linewidth=0.8, label="D (zero field)")
plt.xlabel("Field (G)")
plt.ylabel("Frequency (GHz)")
plt.title("Predicted ODMR peaks vs. field, [100]-aligned field on (100)-cut diamond")
plt.legend()
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'